# Modelo 1

In [2]:
from pathlib import Path
import pandas as pd
import gurobipy as gp
import numpy as np
from gurobipy import GRB


In [3]:
def resolver_modelo_1(F, C, free_vector, imprimir_gurobi=False):
    """
    Resuelve el Modelo 1 de asignación de salas.
    F: Matriz de flujo de estudiantes (I x K)
    C: Matriz de costos/distancias entre salas (R x R)
    """
    # Crear el modelo
    model = gp.Model("Modelo_Homogeneo_Costos")

    print("Shape de C:", C.shape) #C debe ser cuadrada |R| x|R|
    print("Shape de F:", F.shape) #F debe ser |I| x |K|
    print("Largo de free_vector:", len(free_vector))

    print("\nMatriz de costos:") #Muestra la matriz de costoso de este caso
    print(C)

    print("\nMatriz de flujo:") #Muestra la matriz de flujo de este caso
    print(F)

    print("\nVector libre:") #Muestra el vector libre de este caso 
    print(free_vector)
    
    # Esto evita que Gurobi imprima todo su proceso en la consola si no queremos
    if not imprimir_gurobi:
        model.setParam('OutputFlag', 0)

    # 1. CONJUNTOS 
    I = range(len(F))      # Cursos Bloque 1
    K = range(len(F[0]))   # Cursos Bloque 2
    R = range(len(C))      # Salas

    # 2. VARIABLES
    x = model.addVars(I, R, vtype=GRB.BINARY, name="x")
    y = model.addVars(K, R, vtype=GRB.BINARY, name="y")
    z = model.addVars(I, K, R, R, vtype=GRB.BINARY, name="z")

    # 3. FUNCIÓN OBJETIVO
    objective = gp.quicksum(F[i][k] * C[r][s] * z[i,k,r,s] 
                            for i in I for k in K for r in R for s in R)
    model.setObjective(objective, GRB.MINIMIZE)

    # 4. RESTRICCIONES DE ASIGNACIÓN
    model.addConstrs((x.sum(i, '*') == 1 for i in I), "Asig_Bloque1") #Cada curso del bloque 1 debe ser asignado a exactamente una sala
    model.addConstrs((y.sum(k, '*') == 1 for k in K), "Asig_Bloque2") #Cada curso del bloque 2 debe ser asignado a exactamente una sala
    model.addConstrs((x.sum('*', r) <= 1 for r in R), "Ocupacion_R1") #Cada sala puede tener a lo más un curso del bloque 1
    model.addConstrs((y.sum('*', r) <= 1 for r in R), "Ocupacion_R2") #Cada sala puede tener a lo más un curso del bloque 2

    # 5. RESTRICCIONES DE LINEALIZACIÓN (para asegurar que z[i,k,r,s] = 1 solo si x[i,r] = 1 y y[k,s] = 1)
    for i in I:
        for k in K:
            for r in R:
                for s in R:
                    model.addConstr(z[i,k,r,s] <= x[i,r])
                    model.addConstr(z[i,k,r,s] <= y[k,s])
                    model.addConstr(z[i,k,r,s] >= x[i,r] + y[k,s] - 1)

    # 6. OPTIMIZAR
    model.optimize()

    # 7. GUARDAR RESULTADOS EN UN DICCIONARIO
    resultados = {
        "status": model.status,
        "valor_optimo": None,
        "asignacion_b1": [],
        "asignacion_b2": []
    }

    if model.status == GRB.OPTIMAL:
        resultados["valor_optimo"] = model.objVal
        
        # Guardar qué curso fue a qué sala (Bloque 1)
        for i in I:
            for r in R:
                if x[i, r].X > 0.5:
                    resultados["asignacion_b1"].append((f"Curso I{i+1}", f"Sala R{r+1}"))

        # Guardar qué curso fue a qué sala (Bloque 2)
        for k in K:
            for s in R: # Usamos 's' aquí por convención matemática, pero es el mismo conjunto R
                if y[k, s].X > 0.5:
                    resultados["asignacion_b2"].append((f"Curso K{k+1}", f"Sala R{s+1}"))
                    
    # Limpiamos el modelo de la memoria (Buena práctica si corres miles de simulaciones)
    model.dispose()
    
    return resultados

Después de optimizar, Gurobi deja en model.status un código que indica el resultado:

* GRB.OPTIMAL: encontró solución óptima,
* GRB.INFEASIBLE: no existe solución factible,
* GRB.UNBOUNDED: problema no acotado, etc.

In [4]:
def matriz_costos_binaria(salas):
    """
    C_rs = 0 si r = s
           1 si r != s
    """
    n = len(salas)
    C_rs = pd.DataFrame(
        np.ones((n, n), dtype=int) - np.eye(n, dtype=int),
        index=salas,
        columns=salas
    )
    return C_rs


def validar_instancia(F, f_L, C):
    """
    Verifica:
    1) Cada fila suma C considerando libres
    2) Cada columna del bloque 2 suma C
    """
    chequeo_filas = F.sum(axis=1) + f_L
    chequeo_columnas = F.sum(axis=0)

    assert (chequeo_filas == C).all(), (
        "Error: no todas las filas suman C al agregar libres.\n"
        f"{chequeo_filas}"
    )
    assert (chequeo_columnas == C).all(), (
        "Error: no todas las columnas suman C.\n"
        f"{chequeo_columnas}"
    )



"""
def exportar_instancia(instancia, carpeta_salida):
    
    #Exporta los datos a CSV.

    carpeta = Path(carpeta_salida)
    carpeta.mkdir(parents=True, exist_ok=True)

    # Cursos bloque 1
    pd.DataFrame({
        "curso_b1": instancia["I"],
        "tamano": [instancia["C"]] * len(instancia["I"])
    }).to_csv(carpeta / "cursos_bloque1.csv", index=False)

    # Cursos bloque 2
    pd.DataFrame({
        "curso_b2": instancia["K"],
        "tamano": [instancia["C"]] * len(instancia["K"])
    }).to_csv(carpeta / "cursos_bloque2.csv", index=False)

    # Salas
    pd.DataFrame({
        "sala": instancia["R"],
        "capacidad": [instancia["C"]] * len(instancia["R"])
    }).to_csv(carpeta / "salas.csv", index=False)

    # Flujos
    instancia["F"].to_csv(carpeta / "flujos_F.csv")

    # Libres
    instancia["f_L"].to_csv(carpeta / "flujos_libres.csv", header=True)

    # Costos entre salas
    instancia["C_rs"].to_csv(carpeta / "costos_salas.csv")

    # Metadata simple
    with open(carpeta / "resumen.txt", "w", encoding="utf-8") as f:
        f.write(f"C = {instancia['C']}\n")
        f.write(f"Objetivo esperado = {instancia['objetivo_esperado']}\n")
        f.write("Matching esperado:\n")
        for k, i in instancia["matching_esperado"].items():
            f.write(f"  {k} <- {i}\n")

"""

'\ndef exportar_instancia(instancia, carpeta_salida):\n\n    #Exporta los datos a CSV.\n\n    carpeta = Path(carpeta_salida)\n    carpeta.mkdir(parents=True, exist_ok=True)\n\n    # Cursos bloque 1\n    pd.DataFrame({\n        "curso_b1": instancia["I"],\n        "tamano": [instancia["C"]] * len(instancia["I"])\n    }).to_csv(carpeta / "cursos_bloque1.csv", index=False)\n\n    # Cursos bloque 2\n    pd.DataFrame({\n        "curso_b2": instancia["K"],\n        "tamano": [instancia["C"]] * len(instancia["K"])\n    }).to_csv(carpeta / "cursos_bloque2.csv", index=False)\n\n    # Salas\n    pd.DataFrame({\n        "sala": instancia["R"],\n        "capacidad": [instancia["C"]] * len(instancia["R"])\n    }).to_csv(carpeta / "salas.csv", index=False)\n\n    # Flujos\n    instancia["F"].to_csv(carpeta / "flujos_F.csv")\n\n    # Libres\n    instancia["f_L"].to_csv(carpeta / "flujos_libres.csv", header=True)\n\n    # Costos entre salas\n    instancia["C_rs"].to_csv(carpeta / "costos_salas.csv")\n

## Data sintética Poco Movimientos

In [5]:
def generar_test_A(C=30):
    """
    TEST A: casi diagonal, con muy poca dispersión.
    Objetivo esperado: 21

    Estructura:
    - 8 cursos en bloque 1: i1,...,i8
    - 7 cursos en bloque 2: k1,...,k7
    - 8 salas: r1,...,r8

    Intuición de solución óptima:
    k1 con i1, k2 con i2, ..., k7 con i7
    i8 queda mayoritariamente libre.
    """
    I = [f"i{i}" for i in range(1, 9)]
    K = [f"k{k}" for k in range(1, 8)]
    R = [f"r{r}" for r in range(1, 9)]

    # Matriz de flujos
    F = pd.DataFrame(0, index=I, columns=K, dtype=int)

    # Estudiantes libres
    f_L = pd.Series(0, index=I, name="libre", dtype=int)

    # Filas i1,...,i7: flujo dominante al curso "natural"
    F.loc["i1", ["k1", "k2"]] = [27, 2]
    f_L["i1"] = 1

    for j in range(2, 7):  # i2,...,i6
        fila = f"i{j}"
        F.loc[fila, [f"k{j-1}", f"k{j}", f"k{j+1}"]] = [1, 27, 1]
        f_L[fila] = 1

    F.loc["i7", ["k6", "k7"]] = [2, 27]
    f_L["i7"] = 1

    # i8 rellena los déficits de columnas y queda mayoritariamente libre
    # Déficits calculados para que cada columna sume exactamente 30:
    # k1:+2, k3:+1, k4:+1, k5:+1, k7:+2
    F.loc["i8", ["k1", "k3", "k4", "k5", "k7"]] = [2, 1, 1, 1, 2]
    f_L["i8"] = 23

    # Costos binarios entre salas
    C_rs = matriz_costos_binaria(R)

    # Validación
    validar_instancia(F, f_L, C)

    # Objetivo esperado:
    # Total bloque 2 = 7 * 30 = 210
    # Máximo "mismo salón" obvio = 27*7 = 189
    # Costo óptimo esperado = 210 - 189 = 21
    objetivo_esperado = 21

    # Emparejamiento natural esperado
    matching_esperado = {
        "k1": "i1",
        "k2": "i2",
        "k3": "i3",
        "k4": "i4",
        "k5": "i5",
        "k6": "i6",
        "k7": "i7",
    }

    return {
        "I": I,
        "K": K,
        "R": R,
        "C": C,
        "F": F,
        "f_L": f_L,
        "C_rs": C_rs,
        "objetivo_esperado": objetivo_esperado,
        "matching_esperado": matching_esperado,
    }


In [6]:
test_A = generar_test_A(C=30)
print("\nCosto esperado:", test_A["objetivo_esperado"])
print("\nMatching esperado:", test_A["matching_esperado"])


Costo esperado: 21

Matching esperado: {'k1': 'i1', 'k2': 'i2', 'k3': 'i3', 'k4': 'i4', 'k5': 'i5', 'k6': 'i6', 'k7': 'i7'}


In [7]:
resultado_escenario_A=resolver_modelo_1(test_A["F"].values, test_A["C_rs"].values, test_A["f_L"].values, False)
print(f"Costo Total de Movilidad: {resultado_escenario_A['valor_optimo']}")
print(f"Asignación Bloque 1: {resultado_escenario_A['asignacion_b1']}")
print(f"Asignación Bloque 2: {resultado_escenario_A['asignacion_b2']}")

Set parameter Username
Set parameter LicenseID to value 2808412


Academic license - for non-commercial use only - expires 2027-04-16
Shape de C: (8, 8)
Shape de F: (8, 7)
Largo de free_vector: 8

Matriz de costos:
[[0 1 1 1 1 1 1 1]
 [1 0 1 1 1 1 1 1]
 [1 1 0 1 1 1 1 1]
 [1 1 1 0 1 1 1 1]
 [1 1 1 1 0 1 1 1]
 [1 1 1 1 1 0 1 1]
 [1 1 1 1 1 1 0 1]
 [1 1 1 1 1 1 1 0]]

Matriz de flujo:
[[27  2  0  0  0  0  0]
 [ 1 27  1  0  0  0  0]
 [ 0  1 27  1  0  0  0]
 [ 0  0  1 27  1  0  0]
 [ 0  0  0  1 27  1  0]
 [ 0  0  0  0  1 27  1]
 [ 0  0  0  0  0  2 27]
 [ 2  0  1  1  1  0  2]]

Vector libre:
[ 1  1  1  1  1  1  1 23]
Costo Total de Movilidad: 21.0
Asignación Bloque 1: [('Curso I1', 'Sala R8'), ('Curso I2', 'Sala R7'), ('Curso I3', 'Sala R6'), ('Curso I4', 'Sala R5'), ('Curso I5', 'Sala R4'), ('Curso I6', 'Sala R3'), ('Curso I7', 'Sala R2'), ('Curso I8', 'Sala R1')]
Asignación Bloque 2: [('Curso K1', 'Sala R8'), ('Curso K2', 'Sala R7'), ('Curso K3', 'Sala R6'), ('Curso K4', 'Sala R5'), ('Curso K5', 'Sala R4'), ('Curso K6', 'Sala R3'), ('Curso K7', 'Sala R2

## Data sintética Sin Movimiento

In [8]:
def generar_test_B(C=30):
    """
    TEST B: completamente diagonal.
    Objetivo esperado: 0

    Intuición de solución óptima:
    k1 con i1, ..., k7 con i7
    i8 queda totalmente libre.
    """
    I = [f"i{i}" for i in range(1, 9)]
    K = [f"k{k}" for k in range(1, 8)]
    R = [f"r{r}" for r in range(1, 9)]

    # Matriz de flujos
    F = pd.DataFrame(0, index=I, columns=K, dtype=int)

    # Diagonal perfecta en los primeros 7 cursos
    for j in range(1, 8):
        F.loc[f"i{j}", f"k{j}"] = C

    # Estudiantes libres
    f_L = pd.Series(0, index=I, name="libre", dtype=int)
    f_L["i8"] = C

    # Costos binarios entre salas
    C_rs = matriz_costos_binaria(R)

    # Validación
    validar_instancia(F, f_L, C)

    objetivo_esperado = 0

    matching_esperado = {
        "k1": "i1",
        "k2": "i2",
        "k3": "i3",
        "k4": "i4",
        "k5": "i5",
        "k6": "i6",
        "k7": "i7",
    }

    return {
        "I": I,
        "K": K,
        "R": R,
        "C": C,
        "F": F,
        "f_L": f_L,
        "C_rs": C_rs,
        "objetivo_esperado": objetivo_esperado,
        "matching_esperado": matching_esperado,
    }


In [9]:

test_B = generar_test_B(C=30)
print("\nCosto esperado:", test_B["objetivo_esperado"])
print("\nMatching esperado:", test_B["matching_esperado"])



Costo esperado: 0

Matching esperado: {'k1': 'i1', 'k2': 'i2', 'k3': 'i3', 'k4': 'i4', 'k5': 'i5', 'k6': 'i6', 'k7': 'i7'}


In [11]:
resultado_escenario_B=resolver_modelo_1(test_B["F"].values, test_B["C_rs"].values, test_B["f_L"].values, False)
print(f"Costo Total de Movilidad: {resultado_escenario_B['valor_optimo']}")
print(f"Asignación Bloque 1: {resultado_escenario_B['asignacion_b1']}")
print(f"Asignación Bloque 2: {resultado_escenario_B['asignacion_b2']}")

Shape de C: (8, 8)
Shape de F: (8, 7)
Largo de free_vector: 8

Matriz de costos:
[[0 1 1 1 1 1 1 1]
 [1 0 1 1 1 1 1 1]
 [1 1 0 1 1 1 1 1]
 [1 1 1 0 1 1 1 1]
 [1 1 1 1 0 1 1 1]
 [1 1 1 1 1 0 1 1]
 [1 1 1 1 1 1 0 1]
 [1 1 1 1 1 1 1 0]]

Matriz de flujo:
[[30  0  0  0  0  0  0]
 [ 0 30  0  0  0  0  0]
 [ 0  0 30  0  0  0  0]
 [ 0  0  0 30  0  0  0]
 [ 0  0  0  0 30  0  0]
 [ 0  0  0  0  0 30  0]
 [ 0  0  0  0  0  0 30]
 [ 0  0  0  0  0  0  0]]

Vector libre:
[ 0  0  0  0  0  0  0 30]
Costo Total de Movilidad: 0.0
Asignación Bloque 1: [('Curso I1', 'Sala R7'), ('Curso I2', 'Sala R6'), ('Curso I3', 'Sala R5'), ('Curso I4', 'Sala R4'), ('Curso I5', 'Sala R3'), ('Curso I6', 'Sala R2'), ('Curso I7', 'Sala R1'), ('Curso I8', 'Sala R8')]
Asignación Bloque 2: [('Curso K1', 'Sala R7'), ('Curso K2', 'Sala R6'), ('Curso K3', 'Sala R5'), ('Curso K4', 'Sala R4'), ('Curso K5', 'Sala R3'), ('Curso K6', 'Sala R2'), ('Curso K7', 'Sala R1')]


## Data Sintética Con Movimiento

In [12]:
I = ["i1", "i2", "i3", "i4"]
K = ["k1", "k2", "k3"]
R = ["r1", "r2", "r3", "r4"]

C = 20

# Flujos entre bloque 1 y bloque 2
F = pd.DataFrame([
    [18,  1,  0],
    [ 1, 18,  0],
    [ 0,  1, 18],
    [ 1,  0,  2],
], index=I, columns=K)

# Estudiantes libres
f_L = pd.Series([1, 1, 1, 17], index=I, name="libre")

# Matriz de costos binaria entre salas
C_rs = pd.DataFrame(
    np.ones((len(R), len(R)), dtype=int) - np.eye(len(R), dtype=int),
    index=R, columns=R
)

In [13]:
# DATOS SINTÉTICOS DE JUPYTER NOTEBOOK
# Ruta a la carpeta data/modelo1
base_dir = Path("data/modelo1")

# Leer archivos
C_df = pd.read_csv(base_dir / "cost_matrix.csv", index_col=0)
F_df = pd.read_csv(base_dir / "flow_matrix.csv", index_col=0)
L_df = pd.read_csv(base_dir / "free_vector.csv")


# Convertir a estructuras numéricas
C = C_df.to_numpy()
F = F_df.to_numpy()
free_vector = L_df["free_students"].to_numpy()


resultado_escenario=resolver_modelo_1(F, C, free_vector, False)
print(f"Costo Total de Movilidad: {resultado_escenario['valor_optimo']}")
print(f"Asignación Bloque 1: {resultado_escenario['asignacion_b1']}")
print(f"Asignación Bloque 2: {resultado_escenario['asignacion_b2']}")

Shape de C: (8, 8)
Shape de F: (8, 6)
Largo de free_vector: 8

Matriz de costos:
[[0 1 1 1 1 1 1 1]
 [1 0 1 1 1 1 1 1]
 [1 1 0 1 1 1 1 1]
 [1 1 1 0 1 1 1 1]
 [1 1 1 1 0 1 1 1]
 [1 1 1 1 1 0 1 1]
 [1 1 1 1 1 1 0 1]
 [1 1 1 1 1 1 1 0]]

Matriz de flujo:
[[ 1  0  0  0  2  0]
 [ 6 24  0  0  0  0]
 [14  0  0  0  0 16]
 [ 0  0  8  0  0  3]
 [ 1  0  0  0 18 11]
 [ 0  6  0  0 10  0]
 [ 8  0 22  0  0  0]
 [ 0  0  0 30  0  0]]

Vector libre:
[27  0  0 19  0 14  0  0]
Costo Total de Movilidad: 69.0
Asignación Bloque 1: [('Curso I1', 'Sala R8'), ('Curso I2', 'Sala R7'), ('Curso I3', 'Sala R6'), ('Curso I4', 'Sala R5'), ('Curso I5', 'Sala R4'), ('Curso I6', 'Sala R3'), ('Curso I7', 'Sala R2'), ('Curso I8', 'Sala R1')]
Asignación Bloque 2: [('Curso K1', 'Sala R6'), ('Curso K2', 'Sala R7'), ('Curso K3', 'Sala R2'), ('Curso K4', 'Sala R1'), ('Curso K5', 'Sala R4'), ('Curso K6', 'Sala R5')]
